# Exercise: Feed Clean Documents into Your RAG Pipeline

In the walkthrough you built the RAG pipeline by hand on text pulled straight from a PDF reader. Real documents are messier than that, and the quality of the text you feed in directly shapes the quality of what you retrieve.

In this exercise you will reuse the previous pipeline but we convert the document to Markdown with
[MarkItDown](https://github.com/microsoft/markitdown), clean it, and then **measure** how much that cleaning changes what the pipeline retrieves and answers.

## Learning Objectives

By completing this exercise, you will be able to:

- Convert a PDF to Markdown with MarkItDown and inspect the raw conversion.
- Write cleaning rules that remove common extraction artifacts.
- Measure the effect of cleaning on retrieval with similarity scores and chunk overlap.

## Setup

Load your credentials. The Groq client reads `GROQ_API_KEY` from your `.env` file.

In [1]:
from dotenv import load_dotenv

load_dotenv()

PDF = "documents/react_paper.pdf"
QUERY = "What is the ReAct framework and how does it combine reasoning and acting?"

## Provided: the RAG pipeline from the walkthrough

Run this cell as is. It gives you two things:

- `run_rag(text, query)`, which chunks a body of text, embeds it, retrieves the most
  relevant chunks, and generates an answer. It returns the `answer`, the retrieved
  `chunks`, and their cosine similarity `scores`.
- `retrieval_overlap(chunks_a, chunks_b)`, the Jaccard overlap between two sets of
  retrieved chunks (0 means completely different, 1 means identical).

You do not need to change anything here.

In [2]:
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


def chunk_text(text, size=800, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


def retrieve(query, chunks, chunk_vectors, k=3):
    query_vector = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ query_vector
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]


def generate_answer(query, context):
    prompt = (
        "Answer the question using only the context below.\n\n"
        "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {query}\nAnswer:"
    )
    response = Groq().chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


def run_rag(text, query, k=3):
    """Chunk, embed, retrieve, and generate. Returns answer, chunks, and scores."""
    chunks = chunk_text(text)
    chunk_vectors = embedder.encode(chunks, normalize_embeddings=True)
    hits = retrieve(query, chunks, chunk_vectors, k=k)
    context = [chunk for chunk, _ in hits]
    return {
        "answer": generate_answer(query, context),
        "chunks": context,
        "scores": [score for _, score in hits],
    }


def retrieval_overlap(chunks_a, chunks_b):
    """Jaccard overlap between two lists of retrieved chunks (0 to 1)."""
    a, b = set(chunks_a), set(chunks_b)
    return len(a & b) / len(a | b) if (a or b) else 0.0

/Users/fritzvonscherzer/Documents/AIPM/aipm-rag-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Exercise 1: Convert the document to Markdown

Use MarkItDown to convert the PDF to a Markdown string.

Hints:

- The API is `MarkItDown().convert(path).text_content`.
- After converting, print the first 1000 characters and look closely. Real PDF
  extractions often have missing spaces, repeated page headers, and stray line breaks.
  Note the problems you see: you will fix them next.

In [3]:
from markitdown import MarkItDown


def to_markdown(path):
    """Convert a document to a Markdown string with MarkItDown."""
    return MarkItDown().convert(path).text_content

In [4]:
raw_text = to_markdown(PDF)
print(raw_text[:1000])

# Problems spotted:
# - words run together without spaces ("Whilelargelanguagemodels...")
# - the conference header "PublishedasaconferencepaperatICLR2023" repeats on every page
# - words split across lines with a hyphen ("Con-\nsider...")
# - stray page numbers and tiny fragments from figures and tables

PublishedasaconferencepaperatICLR2023
REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
ShunyuYao∗*,1,JeffreyZhao2,DianYu2,NanDu2,IzhakShafran2,KarthikNarasimhan1,YuanCao2
1DepartmentofComputerScience,PrincetonUniversity
2GoogleResearch,Brainteam
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
Whilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance
across tasks in language understanding and interactive decision making, their
abilitiesforreasoning(e.g. chain-of-thoughtprompting)andacting(e.g. action
plangeneration)haveprimarilybeenstudiedasseparatetopics. Inthispaper,we
exploretheuseofLLMstogeneratebothreasoningtracesandtask-specificactions
inaninterleavedmanner,allowingforgreatersynergybetweenthetwo: reasoning
traces help the model induce, track, and update action plans as well as handle
exceptions,whileactionsallowittointerfacewithandgatheradditionalinformation
fromexternalsourcessuchasknowledgebasesorenvironments. W

## Exercise 2: Clean the Markdown

Write a cleaning function that fixes the problems you spotted. Decide which rules
matter for this document.

Ideas to consider:

- Collapse runs of blank lines and strip trailing whitespace.
- Drop lines that are just a page number or the repeated conference header.
- Remove very short lines that are page furniture rather than content.
- Rejoin words split across a line break by a hyphen.

Store the result as `clean_text` and print the character count before and after so you
can see how much you removed.

In [5]:
import re

HEADER = re.compile(r"publishedasaconferencepaperaticlr2023", re.IGNORECASE)


def clean_markdown(text):
    """Return a cleaned version of the Markdown text."""
    # rejoin words split across a line break by a hyphen
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    lines = []
    for line in text.split("\n"):
        line = line.rstrip()
        stripped = line.strip()
        if HEADER.search(stripped):  # repeated conference header
            continue
        if re.fullmatch(r"\d{1,3}", stripped):  # bare page number
            continue
        if stripped and len(stripped) < 4:  # page furniture / figure fragments
            continue
        lines.append(line)
    text = "\n".join(lines)
    # collapse runs of blank lines
    return re.sub(r"\n{3,}", "\n\n", text).strip()

In [6]:
clean_text = clean_markdown(raw_text)
print(f"before: {len(raw_text)} characters, after: {len(clean_text)} characters")

before: 205117 characters, after: 203469 characters


## Exercise 3: Measure the effect of cleaning

This is the payoff. Run the provided pipeline twice on the same question, once with
`raw_text` and once with `clean_text`, then compare them with two metrics.

**Metric 1: mean retrieval score.** Average the `scores` of the retrieved chunks for
each run. A higher mean suggests the retrieved chunks match the query more closely.

**Metric 2: retrieval overlap.** Use `retrieval_overlap(...)` on the two `chunks`
lists. A low overlap means cleaning changed *which* chunks were retrieved.

> Read the scores as a signal, not a verdict. The raw and cleaned runs have different
> chunk sets, so the numbers are not a perfect like-for-like comparison. Always read the
> retrieved chunks and the answer alongside the metrics.

Tasks:

- Call `run_rag` on `raw_text` and on `clean_text`.
- Print the mean retrieval score for each run and the overlap between them.
- Print both answers, then write a sentence on what changed and why.

In [7]:
import statistics

raw_result = run_rag(raw_text, QUERY)
clean_result = run_rag(clean_text, QUERY)

print("Mean retrieval score (raw):  ", round(statistics.mean(raw_result["scores"]), 3))
print("Mean retrieval score (clean):", round(statistics.mean(clean_result["scores"]), 3))
print(
    "Retrieval overlap:           ",
    round(retrieval_overlap(raw_result["chunks"], clean_result["chunks"]), 3),
)

print("\n--- RAW ANSWER ---\n", raw_result["answer"])
print("\n--- CLEAN ANSWER ---\n", clean_result["answer"])

Mean retrieval score (raw):   0.51
Mean retrieval score (clean): 0.559
Retrieval overlap:            0.0

--- RAW ANSWER ---
 **ReAct** is a framework that lets a large language model *think* and *act* in the same sequence.  
- The model alternates between **thoughts** (internal reasoning written in natural language) and **actions** (the concrete step it takes in the task).  
- A ReAct prompt is simply the model’s own thoughts followed by the action it will perform; no special formatting or example selection is required.  
- Because the thought space is flexible and the thought–action pattern is generic, ReAct can be applied to many kinds of tasks—question answering, fact verification, text‑based games, web navigation, etc.  
- The framework relies on the model’s ability to retrieve useful external knowledge; a non‑informative search often causes the reasoning loop to fail.  
- Compared with simpler “IM‑style” prompting, ReAct’s internal reasoning gives it a higher success rate and bet

**What changed:** the mean retrieval score rose from 0.510 to 0.559 and the overlap was 0.0, so cleaning changed *which* chunks were retrieved. Removing the repeated header and page numbers shifted every fixed-size chunk boundary, so the two runs are not like-for-like. Both answers describe ReAct as interleaving thoughts and actions, so the answer was fairly robust even though the chunks behind it were different.

## Exercise 4 (stretch): Score the answers with an LLM judge

Similarity scores tell you about retrieval, not about the final answer. A common way to
score answer quality without a reference answer is to use the model as a judge.

Write a function that asks the model to rate, from 1 to 5, how well an answer is
supported by the context it was given (its faithfulness). Run it for both the raw and
the cleaned answer and compare.

Hint: build a prompt that includes the question, the context, and the answer, and ask
the model to reply with a single number. Reuse the `Groq` client from the provided cell.

In [8]:
def judge_faithfulness(question, context, answer):
    """Ask the model to rate 1 to 5 how well the answer is supported by the context."""
    prompt = (
        "Rate from 1 to 5 how well the answer is supported by the context, "
        "with no invented facts (5 = fully supported).\n\n"
        "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {question}\n\n"
        f"Answer: {answer}\n\nReply with a single integer from 1 to 5."
    )
    response = Groq().chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    content = response.choices[0].message.content or ""
    match = re.search(r"[1-5]", content)
    return int(match.group()) if match else None

In [9]:
raw_faith = judge_faithfulness(QUERY, raw_result["chunks"], raw_result["answer"])
clean_faith = judge_faithfulness(QUERY, clean_result["chunks"], clean_result["answer"])
print("Faithfulness (raw):  ", raw_faith)
print("Faithfulness (clean):", clean_faith)

Faithfulness (raw):   5
Faithfulness (clean): 5


## Conclusion and Reflection

- Which cleaning rule made the biggest difference to the mean retrieval score or the
  overlap, and why?
- Did cleaning change the answer, or only the chunks behind it? What does that tell you
  about how sensitive this question is to noise?
- The mean score is an imperfect metric. What would you need in order to measure answer
  quality properly, and which tools (for example RAGAS) formalise this?

**My answers**

- The biggest effect came from dropping the repeated header and page furniture, but with fixed-size chunking any removal moves the boundaries, so the overlap of 0.0 is mostly a chunking effect. The missing spaces between words are the worst artifact left, and none of my rules fix them.
- Cleaning changed the chunks far more than the answer. The question is broad and ReAct is described in many places, so it tolerates noise. A narrow factual question would be more sensitive.
- To measure answer quality properly you need labelled question/answer pairs or reference contexts. RAGAS formalises faithfulness, answer relevance, context precision and recall on top of that.